In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from LSTM_databuilder import AssistSequenceDataset
from LSTM_model_train import AssistLSTM

### video completion status


'''
Frame-level data
    ↓
Sliding window dataset
    ↓
LSTM(seq2one)
    ↓
urgency_pred + type_pred
    ↓
threshold
    ↓
assist trigger
'''

In [14]:
#device 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### hyper param

In [ ]:
epochs = 20
batch_size = 32
lr = 1e-3
window_size = 120
stride = 30
num_workers = 4

### load data

In [7]:
npz_path = r"C:\Users\loy49\Desktop\REPO\Multi_Head_HRC\data\dataset\dataset_lift.npz"

In [ ]:
dataset = AssistSequenceDataset(npz_path=npz_path,
                                window_size=120,
                                stride=30,
                                predict_offset=0,
                                mode="seq2one",
                                use_type=True)
loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=(device.type == "cuda"), drop_last=False)
for x, y in loader:
    print(type(x))
    print(type(y))
    break

<class 'torch.Tensor'>
<class 'list'>


### model

In [11]:
# get one batch to infer feature_dim and num_types
x_sample, (y_urg_sample, y_type_sample) = next(iter(loader))
input_dim = x_sample.shape[2]

num_types = int(torch.max(y_type_sample).item()) + 1

In [ ]:
model = AssistLSTM(
    input_dim=input_dim,
    hidden_dim=128,
    num_types=num_types
).to(device)

optimizer = optim.Adam(model.parameters(), lr=lr)

ce_loss = nn.CrossEntropyLoss()
mse_loss = nn.MSELoss()

lambda_type = 1.0
lambda_urgency = 1.0

model.train()

AssistLSTM(
  (lstm): LSTM(132, 128, batch_first=True)
  (type_head): Linear(in_features=128, out_features=1, bias=True)
  (urgency_head): Linear(in_features=128, out_features=1, bias=True)
)

### train

In [13]:
#train loop
for epoch in range(epochs):

    total_loss = 0
    total_type_correct = 0
    total_samples = 0

    for x, (y_urgency, y_type) in loader:

        optimizer.zero_grad()

        type_logits, urgency_pred = model(x)

        loss_type = ce_loss(type_logits, y_type)
        loss_urgency = mse_loss(urgency_pred, y_urgency)

        loss = (
            lambda_type * loss_type +
            lambda_urgency * loss_urgency
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # type accuracy
        preds = torch.argmax(type_logits, dim=1)
        total_type_correct += (preds == y_type).sum().item()
        total_samples += y_type.size(0)

    acc = total_type_correct / total_samples

    print(f"Epoch {epoch+1}/{epochs} "
            f"| Loss: {total_loss/len(loader):.4f} "
            f"| Type Acc: {acc:.4f}")

torch.save(model.state_dict(), "assist_model.pth")
print("Model saved.")


Epoch 1/20 | Loss: 0.1749 | Type Acc: 1.0000
Epoch 2/20 | Loss: 0.1452 | Type Acc: 1.0000
Epoch 3/20 | Loss: 0.1267 | Type Acc: 1.0000
Epoch 4/20 | Loss: 0.1182 | Type Acc: 1.0000
Epoch 5/20 | Loss: 0.1170 | Type Acc: 1.0000
Epoch 6/20 | Loss: 0.1187 | Type Acc: 1.0000
Epoch 7/20 | Loss: 0.1197 | Type Acc: 1.0000
Epoch 8/20 | Loss: 0.1188 | Type Acc: 1.0000
Epoch 9/20 | Loss: 0.1162 | Type Acc: 1.0000
Epoch 10/20 | Loss: 0.1127 | Type Acc: 1.0000
Epoch 11/20 | Loss: 0.1091 | Type Acc: 1.0000
Epoch 12/20 | Loss: 0.1063 | Type Acc: 1.0000
Epoch 13/20 | Loss: 0.1043 | Type Acc: 1.0000
Epoch 14/20 | Loss: 0.1027 | Type Acc: 1.0000
Epoch 15/20 | Loss: 0.1004 | Type Acc: 1.0000
Epoch 16/20 | Loss: 0.0970 | Type Acc: 1.0000
Epoch 17/20 | Loss: 0.0940 | Type Acc: 1.0000
Epoch 18/20 | Loss: 0.0905 | Type Acc: 1.0000
Epoch 19/20 | Loss: 0.0848 | Type Acc: 1.0000
Epoch 20/20 | Loss: 0.0775 | Type Acc: 1.0000
Model saved.
